# Week 51, Day 3: Union–find

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w051d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
parent = {x: x for x in "abcdef"}
size = {x: 1 for x in "abcdef"}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]        # halve the path as we go
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return False
    if size[ra] < size[rb]:
        ra, rb = rb, ra
    parent[rb], size[ra] = ra, size[ra] + size[rb]
    return True

print([union(a, b) for a, b in ["ab", "cd", "bd", "ac", "ef"]])
print(len({find(x) for x in "abcdef"}), size[find("a")])

In [ ]:
import random

parent, size, depth_seen = {}, {}, 0
def find(x):
    d = 0
    while parent[x] != x:
        x, d = parent[x], d + 1
    return x, d

n = 4096
for i in range(n):
    parent[i], size[i] = i, 1
rng = random.Random(1)
for _ in range(20_000):
    (ra, _), (rb, _) = find(rng.randrange(n)), find(rng.randrange(n))
    if ra != rb:
        if size[ra] < size[rb]:
            ra, rb = rb, ra
        parent[rb], size[ra] = ra, size[ra] + size[rb]
print("deepest:", max(find(i)[1] for i in range(n)), "and log2(n) =", n.bit_length() - 1)

In [ ]:
class Weighted:
    def __init__(self):
        self.parent, self.diff = {}, {}

    def find(self, x):
        if x not in self.parent:
            self.parent[x], self.diff[x] = x, 0
        path = []
        while self.parent[x] != x:
            path.append(x)
            x = self.parent[x]
        root, total = x, 0
        for v in reversed(path):              # from nearest the root outwards
            total += self.diff[v]
            self.parent[v], self.diff[v] = root, total
        return root

    def offset(self, x):
        self.find(x)
        return self.diff[x]

    def relate(self, a, b, k):
        """Record value(a) - value(b) = k. Returns False if it contradicts earlier facts."""
        ra, rb = self.find(a), self.find(b)
        da, db = self.diff[a], self.diff[b]
        if ra == rb:
            return da - db == k
        self.parent[rb], self.diff[rb] = ra, da - db - k
        return True

    def difference(self, a, b):
        if self.find(a) != self.find(b):
            return None
        return self.offset(a) - self.offset(b)

w = Weighted()
print(w.relate("parcel", "box", 3), w.relate("box", "crate", -1), w.difference("parcel", "crate"))
print(w.relate("crate", "parcel", -2), w.relate("crate", "parcel", 5), w.difference("parcel", "lamp"))

In [ ]:
parent, parity = {}, {}

def find(x):
    if x not in parent:
        parent[x], parity[x] = x, 0
    path = []
    while parent[x] != x:
        path.append(x)
        x = parent[x]
    total = 0
    for v in reversed(path):
        total ^= parity[v]
        parent[v], parity[v] = x, total
    return x

def tell(a, b, opposite):
    ra, rb = find(a), find(b)
    want = 1 if opposite else 0
    if ra == rb:
        return (parity[a] ^ parity[b]) == want
    parent[rb], parity[rb] = ra, parity[a] ^ parity[b] ^ want
    return True

facts = [("ana", "ben", True), ("ben", "cy", True), ("ana", "cy", False), ("cy", "ana", True)]
print([tell(*f) for f in facts])

In [ ]:
def pieces_after_closures(towns, roads, closures):
    parent = {t: t for t in towns}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    count = [len(towns)]                           # a one-item list the inner function can change
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            count[0] -= 1
    closed = set(closures)
    for a, b in roads:
        if (a, b) not in closed:
            union(a, b)
    answers = [count[0]]                           # after every closure
    for a, b in reversed(closures[1:]):            # undo closures, last first
        union(a, b)
        answers.append(count[0])
    return answers[::-1]                           # after closure 1, 2, ...

towns = "ABCDE"
roads = [("A", "B"), ("B", "C"), ("C", "D"), ("D", "E"), ("A", "C")]
print(pieces_after_closures(towns, roads, [("B", "C"), ("A", "C"), ("D", "E")]))

In [ ]:
import random

def percolation_point(n, seed):
    parent = list(range(n * n + 2))
    top, bottom = n * n, n * n + 1
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        parent[find(a)] = find(b)
    open_ = [False] * (n * n)
    order = list(range(n * n))
    random.Random(seed).shuffle(order)
    for count, site in enumerate(order, start=1):
        open_[site] = True
        r, c = divmod(site, n)
        if r == 0:
            union(site, top)
        if r == n - 1:
            union(site, bottom)
        for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):
            if 0 <= nr < n and 0 <= nc < n and open_[nr * n + nc]:
                union(site, nr * n + nc)
        if find(top) == find(bottom):
            return count / (n * n)

trials = [percolation_point(40, seed) for seed in range(20)]
print(round(sum(trials) / len(trials), 3))

> **Key idea.** Union by size keeps trees O(log n) deep (a vertex's tree doubles each time it gets deeper); with path compression or halving, each operation costs O(α(n)), effectively constant. Weighted union-find stores an offset on each link, sums offsets during find (fixing them as it compresses), links roots with the offset that makes a new fact true, and reports a contradiction when two items already in one group disagree; ratios work the same way with multiplication. Parity union-find uses XOR for same-or-opposite facts and detects odd cycles online. Deletions known in advance are handled by processing them backwards as unions. Percolation joins open sites with virtual top and bottom sites, so "does it percolate?" is one comparison of roots.

### Exercise: Groups as they grow

Write `UnionFind(items)` with `find` (using path halving, no recursion), `union`, `size_of(x)` and `largest` (an attribute: the size of the largest group, kept up to date in O(1) per union). Then `largest_after_each(items, pairs)`: the largest group size after each union.

In [ ]:
class UnionFind:
    def __init__(self, items):
        ___

def largest_after_each(items, pairs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_growth():\n    \"growth\"\n    assert largest_after_each(\"abcdef\", [\"ab\", \"cd\", \"bd\", \"ac\", \"ef\"]) == [2, 2, 4, 4, 4]\n    uf = UnionFind(range(5))\n    assert uf.largest == 1 and uf.union(0, 1) and not uf.union(1, 0) and uf.size_of(1) == 2\n    rng = random.Random(1)\n    items = list(range(40))\n    pairs = [tuple(rng.sample(items, 2)) for _ in range(60)]\n    got = largest_after_each(items, pairs)\n    groups = {i: {i} for i in items}\n    for (a, b), g in zip(pairs, got):\n        if groups[a] is not groups[b]:\n            merged = groups[a] | groups[b]\n            for x in merged:\n                groups[x] = merged\n        assert g == max(len(s) for s in groups.values())")

### Exercise: Ratios

Facts arrive as `(a, b, k)` meaning a = k × b (k > 0). Write `Ratios` with `tell(a, b, k)` (returning `False`, and changing nothing, if the fact contradicts what's known, comparing with a relative tolerance of 1e-9) and `ask(a, b)`: the ratio a / b, or `None` if it can't be known. Store a multiplicative factor on each link (value(x) / value(parent)), and keep `find` free of recursion.

In [ ]:
class Ratios:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math, random\ndef test_ratios():\n    \"ratios\"\n    r = Ratios()\n    assert r.tell(\"km\", \"m\", 1000) and r.tell(\"m\", \"cm\", 100) and math.isclose(r.ask(\"km\", \"cm\"), 100000)\n    assert math.isclose(r.ask(\"cm\", \"km\"), 1e-5) and r.ask(\"km\", \"kg\") is None and r.ask(\"m\", \"m\") == 1\n    assert not r.tell(\"cm\", \"km\", 2) and math.isclose(r.ask(\"cm\", \"km\"), 1e-5)\n    assert r.tell(\"mile\", \"km\", 1.609344) and math.isclose(r.ask(\"mile\", \"cm\"), 160934.4)\n    rng = random.Random(2)\n    vals = {f\"x{i}\": rng.uniform(0.5, 4) for i in range(30)}\n    r = Ratios()\n    names = list(vals)\n    for _ in range(60):\n        a, b = rng.sample(names, 2)\n        assert r.tell(a, b, vals[a] / vals[b])\n    for _ in range(100):\n        a, b = rng.sample(names, 2)\n        q = r.ask(a, b)\n        assert q is None or math.isclose(q, vals[a] / vals[b], rel_tol=1e-9)\n    chain = Ratios()\n    for i in range(5000):\n        chain.tell(i, i + 1, 1.0)\n    assert chain.ask(0, 5000) == 1.0")

### Exercise: Same team, other team

Facts arrive as lines `ana same ben` or `ana opposite cy`. Write `check(facts)`: process them in order with parity union-find, returning the 1-based number of the first fact that contradicts the earlier ones, or `None`. Then `teams(facts)` (only for consistent facts): a list of pairs of sorted lists `(side_a, side_b)`, one pair per connected group, where `side_a` holds the group's alphabetically first person; the list sorted by that person.

In [ ]:
def check(facts):
    ___

def teams(facts):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_teams():\n    \"teams\"\n    facts = [\"ana opposite ben\", \"ben opposite cy\", \"ana same cy\", \"cy opposite ana\"]\n    assert check(facts) == 4 and check(facts[:3]) is None\n    assert teams(facts[:3]) == [([\"ana\", \"cy\"], [\"ben\"])]\n    assert teams([\"a same b\", \"c opposite d\", \"e same e\"]) == [([\"a\", \"b\"], []), ([\"c\"], [\"d\"]), ([\"e\"], [])]\n    rng = random.Random(3)\n    for _ in range(100):\n        names = list(\"abcdef\")\n        side = {n: rng.randint(0, 1) for n in names}\n        fs = []\n        for _ in range(8):\n            a, b = rng.sample(names, 2)\n            fs.append(f\"{a} {'same' if side[a] == side[b] else 'opposite'} {b}\")\n        assert check(fs) is None\n        a, b = rng.sample(names, 2)\n        wrong = f\"{a} {'opposite' if side[a] == side[b] else 'same'} {b}\"\n        res = check(fs + [wrong])\n        assert res in (None, len(fs) + 1)")

### Exercise: Roads closing

Towns are joined by roads; then roads close, one at a time, in a known order. Write `pieces_after_each(towns, roads, closures)`: the number of separate pieces after each closure, by starting from the final state and adding the closed roads back in reverse. Then `first_split(towns, roads, closures)`: the 1-based number of the first closure after which some two towns that were connected at the start no longer are, or `None`. Every closure is one of the roads; roads are undirected, and a closure `(b, a)` closes road `(a, b)`.

In [ ]:
def pieces_after_each(towns, roads, closures):
    ___

def first_split(towns, roads, closures):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef pieces(towns, roads):\n    parent = {t: t for t in towns}\n    def find(x):\n        while parent[x] != x:\n            x = parent[x]\n        return x\n    for a, b in roads:\n        parent[find(a)] = find(b)\n    return len({find(t) for t in towns})\ndef test_closures():\n    \"closures\"\n    towns = \"ABCDE\"\n    roads = [(\"A\", \"B\"), (\"B\", \"C\"), (\"C\", \"D\"), (\"D\", \"E\"), (\"A\", \"C\")]\n    cl = [(\"B\", \"C\"), (\"C\", \"A\"), (\"D\", \"E\")]\n    assert pieces_after_each(towns, roads, cl) == [1, 2, 3] and first_split(towns, roads, cl) == 2\n    assert first_split(towns, roads, [(\"A\", \"B\")]) is None and pieces_after_each(\"AB\", [], []) == []\n    rng = random.Random(4)\n    for _ in range(50):\n        ts = list(range(8))\n        rs = list({tuple(sorted(rng.sample(ts, 2))) for _ in range(12)})\n        cl = rng.sample(rs, rng.randint(0, len(rs)))\n        want, left = [], list(rs)\n        for c in cl:\n            left.remove(c)\n            want.append(pieces(ts, left))\n        assert pieces_after_each(ts, rs, cl) == want")

### Exercise: Does it percolate?

A grid is a list of strings, `.` open and `#` blocked. Write `percolates(grid)` with union-find and virtual top and bottom sites (joined up, down, left and right). Then `opening_order(n, order)`: given an n × n grid that starts fully blocked and a list of sites `(row, col)` opened one at a time, the 1-based number of the opening after which the grid first percolates, or `None`.

In [ ]:
def percolates(grid):
    ___

def opening_order(n, order):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import deque\ndef bfs_perc(grid):\n    n, m = len(grid), len(grid[0])\n    q = deque((0, c) for c in range(m) if grid[0][c] == \".\")\n    seen = set(q)\n    while q:\n        r, c = q.popleft()\n        if r == n - 1:\n            return True\n        for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\n            if 0 <= nr < n and 0 <= nc < m and grid[nr][nc] == \".\" and (nr, nc) not in seen:\n                seen.add((nr, nc))\n                q.append((nr, nc))\n    return False\ndef test_percolates():\n    \"percolation\"\n    assert percolates([\".#.\", \"..#\", \"#..\"]) and not percolates([\".#.\", \"#.#\", \"#..\"])\n    assert percolates([\".\"]) and not percolates([\"#\"])\n    rng = random.Random(5)\n    for _ in range(200):\n        n, m = rng.randint(1, 6), rng.randint(1, 6)\n        g = [\"\".join(rng.choice(\"..#\") for _ in range(m)) for _ in range(n)]\n        assert percolates(g) == bfs_perc(g)\n    order = [(0, 0), (2, 2), (1, 0), (1, 1), (2, 1)]\n    assert opening_order(3, order) == 5 and opening_order(3, order[:4]) is None and opening_order(1, [(0, 0)]) == 1")

### Bug hunt: Forgetting the offsets

This weighted union-find compresses paths but doesn't add up the offsets as it re-points vertices at the root, so a vertex re-pointed past its old parent keeps its offset to that old parent: this prints `-3` for box minus crate, which should be `-2`. Sum the offsets along the path before re-pointing, so it prints `-2`.

In [ ]:
parent, diff = {}, {}

def find(x):
    if x not in parent:
        parent[x], diff[x] = x, 0
    path = []
    while parent[x] != x:
        path.append(x)
        x = parent[x]
    for v in path:
        parent[v] = x
    return x

def relate(a, b, k):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb], diff[rb] = ra, diff[a] - diff[b] - k

relate("parcel", "box", 3)      # parcel is 3 more than box
relate("crate", "parcel", -1)   # crate is 1 less than parcel
find("box")
find("crate")
print(diff["box"] - diff["crate"])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"add up the offsets\"\n    expect_output(\"-2\")")

### Bug hunt: The size of a non-root

This union compares the sizes of `a` and `b` themselves rather than of their **roots**, so a big group can end up hung under a single item, and the recorded sizes go wrong: this prints `2`. Compare (and update) the roots' sizes so it prints `4`.

In [ ]:
parent = {x: x for x in "abcd"}
size = {x: 1 for x in "abcd"}

def find(x):
    while parent[x] != x:
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return
    if size[a] < size[b]:
        a, b = b, a
        ra, rb = rb, ra
    parent[rb] = ra
    size[a] += size[b]

union("a", "b")
union("c", "a")
union("d", "b")
print(size[find("a")])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"roots\"\n    expect_output(\"4\")")

### Bug hunt: Joining blocked sites

This percolation check joins each open site to all its neighbours, open or **blocked**, so water seems to flow through rock: this prints `True`. Join only open neighbours so it prints `False`.

In [ ]:
grid = [".#", "#."]
n = 2
parent = list(range(n * n + 2))
top, bottom = n * n, n * n + 1

def find(x):
    while parent[x] != x:
        x = parent[x]
    return x

def union(a, b):
    parent[find(a)] = find(b)

for r in range(n):
    for c in range(n):
        if grid[r][c] != ".":
            continue
        site = r * n + c
        if r == 0:
            union(site, top)
        if r == n - 1:
            union(site, bottom)
        for nr, nc in ((r + 1, c), (r, c + 1), (r - 1, c), (r, c - 1)):
            if 0 <= nr < n and 0 <= nc < n:
                union(site, nr * n + nc)
print(find(top) == find(bottom))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"open neighbours only\"\n    expect_output(\"False\")")

### Puzzle: Find with offsets

Arrange the lines into a weighted find that compresses paths and keeps offsets right. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Weighted union-find recording value(a) - value(b) = k.
# Facts: relate("a", "b", 4), relate("b", "c", 1), relate("d", "c", -2).

p = ___   # value("a") - value("c")
q = ___   # value("a") - value("d")
r = ___   # True or False: relate("c", "a", -5) is consistent
s = ___   # True or False: in parity union-find, "x opposite y", "y opposite z", "z opposite x" can all be true
t = ___   # roughly, the fraction of open sites at which a large random grid first percolates (to 3 places)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 5 and q == 7 and r is True, \"p, q and r\"\n    assert s is False and abs(t - 0.593) < 0.001, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: estimating the threshold

Estimate the percolation threshold by experiment. Write `threshold_trial(n, rng)`: open the sites of an n × n grid in a random order (shuffle all n² sites with `rng.shuffle`), using union-find with virtual top and bottom sites, and return the fraction of sites open when the grid first percolates. Then `estimate(n, trials, seed)`: `(mean, standard_deviation)` over `trials` runs, all drawing from one `random.Random(seed)` (use `statistics.mean` and `statistics.stdev`), and `confidence(n, trials, seed)`: the 95% confidence interval mean ± 1.96 × stdev / √trials, as a pair rounded to 3 places. Each trial must be fast: about O(n² α(n²)), so 60 × 60 grids are fine. The checks expect the mean for n = 50 over 30 trials to be within 0.02 of 0.5927.

In [ ]:
import math, random, statistics

def threshold_trial(n, rng):
    ___

def estimate(n, trials, seed):
    ___

def confidence(n, trials, seed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_trial():\n    \"one trial\"\n    rng = random.Random(1)\n    t = threshold_trial(10, rng)\n    assert 0 < t <= 1 and t * 100 == int(round(t * 100))\n    assert threshold_trial(1, random.Random(2)) == 1.0\ndef test_estimate():\n    \"the threshold\"\n    t0 = time.perf_counter()\n    mean, sd = estimate(50, 30, seed=3)\n    assert abs(mean - 0.5927) < 0.02 and 0 < sd < 0.1 and time.perf_counter() - t0 < 10\n    lo, hi = confidence(50, 30, seed=3)\n    assert lo < mean < hi and abs((hi - lo) - 2 * 1.96 * sd / 30 ** 0.5) < 0.002\n    assert estimate(20, 5, seed=9) == estimate(20, 5, seed=9)")

## Warm-up

This version compresses paths a little differently: `parent[x] = parent[parent[x]]` makes every vertex on the path skip to its grandparent (**path halving**). It needs no second loop and gives the same guarantee as full compression.

## Last week, in one example

The essentials from Week 50 Day 4: each group is a tree, `find` follows parents to the root, `union` links one root under the other. With union by size (or by **rank**, an upper bound on height) and path compression or halving, any sequence of m operations on n items costs O(m α(n)), where α is the inverse of the fast-growing Ackermann function. α(n) is at most 4 for any n you could write down in this universe, so each operation is effectively constant.

Why does union by size alone give O(log n) height? A vertex's depth increases only when its tree is hung under a tree at least as big, so each time its depth grows by one, the size of its tree at least **doubles**. A tree of n items can double at most log₂ n times.

That's union by size alone, without compression: the deepest vertex is far below the log₂ n bound.

## Links that carry numbers

Suppose you learn facts like "the parcel weighs 3 kg more than the box" one at a time, and want to answer "how much heavier is the parcel than the crate?" as soon as enough is known. Store, on each link, the **offset** from a vertex to its parent: `diff[x]` means value(x) − value(parent[x]). Then:

- `find(x)` returns the root and the offset from x to the root, the sum of the links along the way. Path compression must **add up** the offsets as it re-points vertices at the root.
- `union(a, b, k)` records value(a) − value(b) = k. With roots ra and rb and offsets da = value(a) − value(ra) and db = value(b) − value(rb), linking rb under ra needs value(rb) − value(ra) = da − db − k.
- If a and b already share a root, the fact is either **consistent** (da − db == k) or a **contradiction**, and you can say so immediately.

This is **weighted** (or "potential") union-find. With multiplication instead of addition, it answers ratio questions ("a is 2.5 times b") in the same way.

## Same side or opposite

A special case with only two values: each item is on one of two **sides**, and facts say "a and b are on the same side" or "on opposite sides" (team members, wiring with polarity, charges). Store a **parity** bit on each link, 0 for same and 1 for opposite, and combine with XOR (`^`) instead of addition. A contradiction is an odd cycle, exactly the reason a graph isn't bipartite (Week 50 Day 2), but detected **online**, the moment the bad fact arrives.

The last fact says Cy and Ana are on opposite sides, but the first two together put them on the same side: union-find spots it at once.

## Running time backwards

Union-find can join groups but not split them. When the problem **removes** edges ("roads close one by one; after each closure, how many towns are cut off?"), there's a neat trick if you know all the removals in advance (an **offline** problem):

1. Start from the graph as it is **after** every removal.
2. Process the removals in **reverse**: each one becomes an **addition**, a `union`.
3. Record the answers as you go, then reverse the list of answers.

The answer after the last removal comes first, from the starting state; each later union rewinds one removal.

(`count` is a one-item list so the inner function can change it; Phase 5 explains the `nonlocal` keyword that does this directly.)

## Percolation

Take an n × n grid of sites, each **open** or **blocked**. The grid **percolates** if water poured on top can reach the bottom through open sites joined up, down, left and right. Open sites one at a time at random: at what fraction of open sites does it first percolate?

Union-find makes each step cheap. The trick is two **virtual** sites: a "top" joined to every open site in the first row, and a "bottom" joined to every open site in the last row. Then "does it percolate?" is just "are top and bottom connected?", one `find` each, instead of a search across the whole grid. For large grids, the answer to the question settles near 0.593, a constant found by simulation (no exact formula is known). The exercise is a classic from Robert Sedgewick and Kevin Wayne's algorithms course.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of facts you know of the form "X is k more than Y" (ages of people you know, heights of buildings) and work out a difference you were never told directly. Tomorrow: minimum spanning trees, the cheapest way to connect everything, with union-find doing the heavy lifting in Kruskal's algorithm.